# Download PR Inline Comments with Kaiaulu (Notebook 5)

This notebook shows how to download pull request inline comments using Kaiaulu’s `download_github_pull_request_comments.Rmd` notebook in the `/vignettes` folder.

### Planned Output

1. A parsed commit-comments CSV saved to `rawdata/github/{owner}/{repo}/{owner}_{repo}_pr_inline_comments.csv` in Kaiaulu

Before starting, complete Steps 1 and 2 in `04_download_commit_comments.ipynb` (confirm working directory and create a GitHub personal access token).

### Step 1: Run download_github_pull_request_comments.Rmd chunks in RStudio

Run the following chunks in **RStudio**. These chunks should already exist in `download_github_pull_request_comments.Rmd`.

### Chunk 1: Set up dependencies

---

```{r warning=FALSE,message=FALSE}
rm(list = ls())
require(kaiaulu)
require(data.table)
require(jsonlite)
require(knitr)
require(magrittr)
require(gt)
require(lubridate)
```

--- 

### Chunk 2: Set required parameters

Replace `kaiaulu.yml` with the `.yml` file for the project you want to process. You created these files in Step 4 of Notebook 3 (`03_scale_config_files.ipynb`).

---

```{r warning=FALSE}
conf <- parse_config("../conf/kaiaulu.yml")
owner <- get_github_owner(conf, "project_key_1") # Has to match github organization (e.g. github.com/sailuh)
repo <- get_github_repo(conf, "project_key_1") # Has to match github repository (e.g. github.com/sailuh/perceive)

# Path you wish to save all raw data.
save_path_pull_request <- get_github_pull_request_path(conf, "project_key_1")
save_path_pr_comments <- get_github_pr_comments_path(conf, "project_key_1")
save_path_issue_or_pr_comments <- get_github_issue_or_pr_comment_path(conf, "project_key_1")
save_path_pr_reviews <- get_github_pr_review_path(conf, "project_key_1")

# Lower API 
save_path_pull_request <- get_github_pull_request_path(conf, "project_key_1")
save_path_pr_commits <- get_github_pr_commits_path(conf, "project_key_1")
save_path_pr_files <- get_github_pr_files_path(conf, "project_key_1")
save_path_pr_reviews <- get_github_pr_review_path(conf, "project_key_1")
save_path_pr_comments <- get_github_pr_comments_path(conf, "project_key_1")

# Create all folder directories
#create_file_directory(conf)
```

---

### Chunk 3: Personal Access Token

Point to the GitHub token created in Step 2 of Notebook 4.

---

```{r Scan GitHub Token}
# your file github_token (a text file) contains the GitHub token API
token <- scan("~/.ssh/github_token",what="character",quiet=TRUE)
```

---

### Chunk 4: Download Pull Request In-Line Code Comments

This chunk downloads PR inline-comment JSON files into `rawdata` in your current working directory. The runtime depends on how many comments the project has.

**IMPORTANT:** This chunk uses `gh_next()` to fetch paginated results and expects `gh` version 1.2.0. If you see a `gh_next()` paging bug (for example, repeated writes to the same page), downgrade to `gh` 1.2.0.

--- 

```{r Collect Comments from Pull Requests, eval = FALSE}
dir.create(save_path_pr_comments, recursive = TRUE, showWarnings = FALSE)
gh_response <- github_api_project_pull_request_inline_comments_refresh(owner, repo, token, save_path_pr_comments)
github_api_iterate_pages(token, gh_response, save_path_pr_comments, prefix="pr_comments")
```

---

### Chunk 5: Parse PR Inline Comments

After all JSON files are downloaded, run the parse chunk for PR inline comments. You should see a table named `inline_comments` in your R environment with columns such as `review_id`, `comment_id`, `html_url`, `created_at`, `updated_at`, `comment_user_login`, `author_association`, `file_path`, `start_line`, `line`, `original_start_line`, `original_line`, `position`, `diff_hunk`, `body`, and `commit_id`.

---

```{r Parse Comments from Pull Requests}
inline_comments <- lapply(list.files(save_path_pr_comments, full.names = TRUE), read_json)
inline_comments <- lapply(inline_comments, github_parse_project_pull_request_inline_comments)
inline_comments <- rbindlist(inline_comments, fill = TRUE)
head(inline_comments,2)  %>%
  gt(auto_align = FALSE) 
```

---

If `fwrite` complains about list/`NULL` columns (common for line/position fields), copy this chunk and run it right after the parse chunk:

```{r Create CSV for Parsed Comments}
as_char_or_na <- function(x) {
  if (is.null(x) || length(x) == 0) return(NA_character_)
  if (is.list(x)) {
    return(vapply(x, function(e) {
      if (is.null(e) || length(e) == 0) NA_character_ else as.character(e[[1]])
    }, character(1)))
  }
  as.character(x)
}
as_int_or_na <- function(x) {
  if (is.null(x) || length(x) == 0) return(NA_integer_)
  if (is.list(x)) {
    return(vapply(x, function(e) {
      if (is.null(e) || length(e) == 0) NA_integer_ else suppressWarnings(as.integer(e[[1]]))
    }, integer(1)))
  }
  suppressWarnings(as.integer(x))
}

for (nm in intersect(c("file_path","diff_hunk","body","html_url","created_at","updated_at","comment_user_login","author_association","commit_id"), names(inline_comments))) {
  if (is.list(inline_comments[[nm]])) inline_comments[[nm]] <- as_char_or_na(inline_comments[[nm]])
}
for (nm in intersect(c("review_id","comment_id","start_line","line","original_start_line","original_line","position"), names(inline_comments))) {
  if (is.list(inline_comments[[nm]])) inline_comments[[nm]] <- as_int_or_na(inline_comments[[nm]])
}

out_csv <- file.path(dirname(save_path_pr_comments), paste0(owner, "_", repo, "_pr_inline_comments.csv"))
data.table::fwrite(inline_comments, out_csv)
cat("Saved:", out_csv, "\n")
```

### Final Output

Final output path:
`rawdata/github/{owner}/{repo}/{owner}_{repo}_pr_inline_comments.csv`

### Next Steps

1. Run Notebooks 4 and 5 for each project configuration (`.yml`) you want to process.
2. Confirm that each run generates the expected commit-comment and PR inline-comment CSV outputs.
3. Use `comment_id` as the join key to transfer sentiment labels to both commit comments and PR inline comments.